# 02 - Data Preparation and Features

This notebook shows how the cleaned data is turned into modeling tables. It uses the same functions as the pipeline, but keeps the intermediate tables visible so the feature choices are easy to review.

The goal is to make the forecast problem explicit: each row should contain one target value, the segment label, calendar context, weather, holiday information, and recent consumption history.


## Setup

We import the feature preparation functions from the project source code. This keeps the notebook aligned with the scripts that generate the final prediction files.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass


import pandas as pd

from group5_energy.pipeline import (
    HALF_TARGET,
    DAILY_TARGET,
    add_history_lags,
    latest_clients,
    load_daily_history,
    load_daily_weather,
    load_half_hourly_history,
    load_holidays,
    load_hourly_weather,
    load_temperatures,
    prepare_daily_frame,
    prepare_half_hourly_frame,
)


## Build the feature tables

Here we load the cleaned history, weather, temperature, and holiday data. Then we create separate feature tables for the half-hourly and daily horizons.

The half-hourly table is used for the 48-hour forecast. The daily table is used for the one-month forecast.


In [ ]:
half_history = load_half_hourly_history()
daily_history = load_daily_history()
weather_hourly = load_hourly_weather()
temperatures = load_temperatures()
weather_daily = load_daily_weather()
holidays = load_holidays()

half_clients = latest_clients(half_history, "DateTime")
daily_clients = latest_clients(daily_history, "Date")

half_features = prepare_half_hourly_frame(half_history, weather_hourly, temperatures, holidays, half_clients)
daily_features = prepare_daily_frame(daily_history, weather_daily, holidays, daily_clients)
half_features = add_history_lags(half_features, HALF_TARGET, "half_hourly")
daily_features = add_history_lags(daily_features, DAILY_TARGET, "daily")

half_features.shape, daily_features.shape

The feature creation step joins outside information to consumption history and adds time-based variables. After this point, the rows are ready for model training and validation.

Keeping the half-hourly and daily tables separate avoids mixing two different forecasting problems. The targets, lags, and time index are different for each horizon.


## Half-hourly features

This sample shows the columns used by the short-term model. We inspect a few rows before training to confirm that the target, weather, holiday flag, time slot, and lag features are present together.


In [ ]:
half_features[[
    "Acorn", "DateTime", "Conso_moy", "temperature", "temperature_half_hour",
    "is_holiday", "half_hour_slot", "lag_48", "lag_336", "rolling_48_mean"
]].head(10)

The half-hourly rows include the current timestamp, the target value, temperature, holiday context, the half-hour slot, and recent history.

The two main lag features have different jobs. The 48-step lag points to the previous day at the same time. The 336-step lag points to the previous week at the same time.


## Daily features

The daily model has fewer rows, so the features are simpler. We still keep weather, calendar, holiday, lag, and rolling average information.


In [ ]:
daily_features[[
    "Acorn", "Date", "Conso_kWh", "temperatureMean", "is_holiday",
    "weekday", "lag_1", "lag_7", "rolling_7_mean"
]].head(10)

The daily table keeps one row per ACORN and date. The one-day lag captures yesterday's level, while the seven-day lag and seven-day rolling mean capture the weekly rhythm.

These features are easy to explain in the report and are useful checks against the baseline models.


## Missing values after feature creation

Lag features create missing values at the beginning of each ACORN series because there is no earlier observation to refer to. We calculate missing rates to separate expected feature gaps from real data quality problems.


In [ ]:
missing_half = half_features.isna().mean().sort_values(ascending=False).head(15)
missing_daily = daily_features.isna().mean().sort_values(ascending=False).head(15)
pd.DataFrame({"half_hourly_missing_rate": missing_half}).join(
    pd.DataFrame({"daily_missing_rate": missing_daily}), how="outer"
)

The missing values are mainly caused by lag and rolling features at the start of each segment history. This is expected and is not the same as a broken timestamp or missing source measurement.

The training pipeline handles these values with model imputers inside the scikit-learn pipeline. That keeps the raw client data unchanged and makes the preprocessing step reproducible.
